# 03 — Deep GRU rating model

Tokenizer + padded sequences, `Embedding` → `GRU` → linear head regression.

This notebook now loads the deep bundle via `app.config.get_deep_paths()` with **Keras first** (`models/deep_rating.keras` + `deep_tokenizer.json`) and **PyTorch fallback** (`models/deep_rating.pt` + `deep_tokenizer.joblib`).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app import config
from scripts.train_and_export import load_and_prepare, train_deep

df = load_and_prepare()
train_deep(df)

In [ ]:
import json

import numpy as np

from app import config
from scripts.train_and_export import _encode

paths = config.get_deep_paths(prefer_keras=True)
texts = ["good quality cable charges fast", "terrible broke in one day"]

if paths["backend"] == "none":
    print(
        "No deep bundle found. Run `python scripts/train_and_export.py` (or train_deep in the cell above) first."
    )
elif paths["backend"] == "keras":
    import tensorflow as tf

    tok = tf.keras.preprocessing.text.tokenizer_from_json(
        paths["tokenizer"].read_text(encoding="utf-8")
    )
    if paths["meta"].exists():
        meta = json.loads(paths["meta"].read_text(encoding="utf-8"))
        max_len = int(meta.get("max_len", 120))
    else:
        max_len = 120

    seqs = tok.texts_to_sequences(texts)
    X = tf.keras.utils.pad_sequences(seqs, maxlen=max_len, padding="post", truncating="post")
    model = tf.keras.models.load_model(paths["model"])
    pred = model.predict(X, verbose=0).reshape(-1)
    print(pred)
else:
    import joblib
    import torch
    from torch import nn

    bundle = torch.load(paths["model"], map_location="cpu")
    tok = joblib.load(paths["tokenizer"])
    vocab = tok["vocab"]
    max_len = tok["max_len"]

    emb_dim = int(bundle["emb_dim"])
    hid_dim = int(bundle["hid_dim"])
    vocab_size = int(bundle["vocab_size"])

    class RatingGRU(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            self.drop = nn.Dropout(0.2)
            self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)
            self.fc = nn.Linear(hid_dim, 1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            e = self.drop(self.emb(x))
            _, h = self.gru(e)
            return self.fc(h[-1]).squeeze(-1)

    model = RatingGRU()
    model.load_state_dict(bundle["state_dict"])
    model.eval()

    X = _encode(texts, vocab, max_len)
    with torch.no_grad():
        pred = model(torch.tensor(X, dtype=torch.long)).numpy()
    print(np.asarray(pred).reshape(-1))